#  Training ViT Transfomer on Fire Detection for outdoor Scenario

## Overview
In this notebook, we train a **SWIN-T** model on a custom dataset designed for detecting **fire scenarios** in **outdoor images**. This dataset represents one of the specific scenarios for our **Mixture of Experts (MoE)** model, where each expert specializes in a different scenario (e.g., fire detection in outdoor, indoor, satellite, or far-field environments).


##  Environment & Setup

In [ ]:
# 1. Install required libraries
!pip install timm torchvision albumentations pycocotools --quiet

##  Data Loading & Preprocessing


In [ ]:
import os
import random
import shutil

# Paths
test_images_dir = 'test/images'
test_labels_dir = 'test/labels'
val_images_dir = 'valid/images'
val_labels_dir = 'valid/labels'

os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

random.seed(42)
test_images = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]
val_images = random.sample(test_images, 300)

for img_file in val_images:
    label_file = os.path.splitext(img_file)[0] + '.txt'
    shutil.move(os.path.join(test_images_dir, img_file), os.path.join(val_images_dir, img_file))
    label_path = os.path.join(test_labels_dir, label_file)
    if os.path.exists(label_path):
        shutil.move(label_path, os.path.join(val_labels_dir, label_file))

print(f"Moved {len(val_images)} images and labels to validation.")


Moved 300 images and their labels to the validation set.


## Checking GPU Availability

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

GPU available: True
Current device: NVIDIA GeForce RTX 3070


## Dataset & Preprocessing

In [ ]:
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import torch

class FireDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, os.path.splitext(self.image_files[idx])[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f.readlines():
                    cls, cx, cy, bw, bh = map(float, line.strip().split())
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)  # Fire class ID

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }

        if self.transform:
            augmented = self.transform(image=image, bboxes=target['boxes'], labels=target['labels'])
            image = augmented['image']
            target['boxes'] = torch.tensor(augmented['bboxes'], dtype=torch.float32)
            target['labels'] = torch.tensor(augmented['labels'], dtype=torch.int64)

        return image, target

    def __len__(self):
        return len(self.image_files)

transform = A.Compose([
    A.Resize(384, 384),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))


## Load Dataloaders

In [ ]:
train_dataset = FireDetectionDataset("train/images", "train/labels", transform)
val_dataset = FireDetectionDataset("valid/images", "valid/labels", transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


## Load Swin Transformer with Detection Head

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.detection.rpn import AnchorGenerator
from timm import create_model

def get_swin_fasterrcnn(num_classes):
    swin = create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
    backbone = BackboneWithFPN(swin, return_layers={2: '0', 3: '1', 4: '2'}, in_channels_list=[192, 384, 768], out_channels=256)
    
    anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256, 512),),
                                       aspect_ratios=((0.5, 1.0, 2.0),))
    
    roi_pooler = torchvision.ops.MultiScaleRoIAlign(featmap_names=['0', '1', '2'],
                                                    output_size=7,
                                                    sampling_ratio=2)
    
    model = FasterRCNN(backbone,
                       num_classes=num_classes,
                       rpn_anchor_generator=anchor_generator,
                       box_roi_pool=roi_pooler)
    
    return model

model = get_swin_fasterrcnn(num_classes=2)  # 1 fire + background
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


## Train the Swin Expert

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs = 10

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for images, targets in tqdm(train_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# Save the trained Swin-T + Faster R-CNN expert for outdoor fire detection
torch.save(model.state_dict(), "swin_frcnn_outdoor.pt")
print("Model saved as swin_frcnn_outdoor.pt")

## Run Inference on Test Images

In [ ]:
model = get_swin_fasterrcnn(num_classes=2)
model.load_state_dict(torch.load("swin_frcnn_outdoor.pt", map_location=device))
model.to(device)
model.eval()
print("Model loaded and ready for inference.")

test_dataset = FireDetectionDataset("test/images", "test/labels", transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

import matplotlib.pyplot as plt

for i, (image, _) in enumerate(test_loader):
    image = image[0].to(device)
    with torch.no_grad():
        prediction = model([image])[0]

    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()

    # Show results
    img_np = image.permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(10, 10))
    plt.imshow(img_np)
    for box, score in zip(boxes, scores):
        if score > 0.5:
            x1, y1, x2, y2 = box
            plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                              edgecolor='red', facecolor='none', linewidth=2))
            plt.text(x1, y1, f"{score:.2f}", color='white', fontsize=12,
                     bbox=dict(facecolor='red', edgecolor='none', pad=1))
    plt.axis('off')
    plt.show()

    if i == 4: break  # Show only first 5
